# 02. 전처리 & 피처 엔지니어링
**핵심 원칙**: 모든 인코더/통계값은 train으로만 fit → test에 transform (Data Leakage 방지)

## 0. 환경 세팅

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


TARGET = '임신 성공 여부'
RANDOM_STATE = 42
COUNT_MAP = {'0회':0,'1회':1,'2회':2,'3회':3,'4회':4,'5회':5,'6회 이상':6}

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')
print(f'train: {train.shape}  |  test: {test.shape}')

## 1. 불필요 컬럼 제거

In [ ]:
# Zero-variance 컬럼 제거
# - 불임 원인 - 여성 요인: 전체가 0
# - 난자 채취 경과일: 값 있으면 전부 0, 결측만 → 완전 무의미
# 주의: PGD/PGS/착상전유전검사는 결측=미시행(0)으로 해석 → 다음 셀에서 처리
DROP_COLS = [
    '불임 원인 - 여성 요인',
    '난자 채취 경과일',
]
train = train.drop(columns=DROP_COLS)
test  = test.drop(columns=DROP_COLS)
print(f'제거 후 → train: {train.shape}  |  test: {test.shape}')
# DI 시술인 경우 배아/난자 관련 결측치 → 0
# 근거: DI는 자궁 내 정자 주입만 하므로 배아 관련 시술 없음 → 실제값이 0
DI_ZERO_COLS = [
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수',
    '이식된 배아 수', '미세주입 배아 이식 수', '저장된 배아 수',
    '미세주입 후 저장된 배아 수', '해동된 배아 수', '해동 난자 수',
    '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
    '난자 혼합 경과일', '배아 이식 경과일', '배아 해동 경과일',
    '동결 배아 사용 여부', '신선 배아 사용 여부', '기증 배아 사용 여부',
    '단일 배아 이식 여부', '임신 시도 또는 마지막 임신 경과 연수',
]
DI_ZERO_COLS = [c for c in DI_ZERO_COLS if c in train.columns]

di_mask_train = train['시술 유형'] == 'DI'
di_mask_test  = test['시술 유형'] == 'DI'

train.loc[di_mask_train, DI_ZERO_COLS] = train.loc[di_mask_train, DI_ZERO_COLS].fillna(0)
test.loc[di_mask_test,   DI_ZERO_COLS] = test.loc[di_mask_test,   DI_ZERO_COLS].fillna(0)

print(f'DI 결측치 0 처리 완료 — train DI 행수: {di_mask_train.sum()}, test DI 행수: {di_mask_test.sum()}')


## 2. 결측치 처리

In [ ]:
# Step 1: PGD/PGS/착상전유전검사 결측 → 0 (미시행)
# 값이 있으면 1(시행), 결측이면 0(미시행)으로 해석
for col in ['착상 전 유전 검사 사용 여부', 'PGD 시술 여부', 'PGS 시술 여부']:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)
print('PGD/PGS/착상전유전검사 결측→0 처리 완료')

In [ ]:
# Step 2: 결측 여부 이진 플래그 생성
# 결측 패턴 자체가 "해당 시술 안 했음"을 의미 → 모델에 추가 정보 제공
FLAG_COLS = [
    '난자 해동 경과일',                     # 결측=동결난자 미사용 (99.4%)
    '배아 해동 경과일',                     # 결측=동결배아 미사용 (84.3%)
    '난자 혼합 경과일',                     # 결측=해당 시술 없음 (21.0%)
    '배아 이식 경과일',                     # 결측=이식 안 함 (17.0%)
    '임신 시도 또는 마지막 임신 경과 연수',  # 결측=첫 시도 가능성 (96.3%)
]
for col in FLAG_COLS:
    train[f'{col}_결측'] = train[col].isnull().astype(int)
    test[f'{col}_결측']  = test[col].isnull().astype(int)
print('결측 플래그 생성 완료')

In [ ]:
# Step 3: 수치형 결측 → train 중앙값 대체
num_cols = train.drop(columns=['ID', TARGET], errors='ignore') \
                .select_dtypes(include=np.number).columns.tolist()
median_vals = train[num_cols].median()          # train으로만 계산
train[num_cols] = train[num_cols].fillna(median_vals)
test[num_cols]  = test[num_cols].fillna(median_vals)  # train 통계 적용
print(f'수치형 결측 처리 완료 | 남은 결측 → train: {train[num_cols].isnull().sum().sum()}')

In [ ]:
# Step 4: 범주형 결측 → train 최빈값 대체
cat_cols = train.drop(columns=['ID', TARGET], errors='ignore') \
               .select_dtypes(include='object').columns.tolist()
mode_vals = train[cat_cols].mode().iloc[0]      # train으로만 계산
train[cat_cols] = train[cat_cols].fillna(mode_vals)
test[cat_cols]  = test[cat_cols].fillna(mode_vals)
print(f'범주형 결측 처리 완료 | 전체 남은 결측 → train: {train.isnull().sum().sum()}  test: {test.isnull().sum().sum()}')

## 3. 피처 엔지니어링

In [ ]:
def add_features(df):
    df = df.copy()

    # 1. 포배기 이식 관련
    # 5일 배양 성공률 40.4% vs 평균 25.8% → 가장 강한 단일 피처
    df['포배기_이식']   = (df['배아 이식 경과일'] == 5).astype(float)
    df['장기배양_이식'] = (df['배아 이식 경과일'] >= 4).astype(float)

    # 2. 난소 반응 관련
    # 저반응(≤3개) 성공률 9.9%, 양호(10~15개) 32.4%
    df['저반응_여부'] = (df['수집된 신선 난자 수'] <= 3).astype(float)
    df['고반응_여부'] = (df['수집된 신선 난자 수'] > 15).astype(float)
    df['성숙_난자율'] = df['혼합된 난자 수'] / (df['수집된 신선 난자 수'] + 1)

    # 3. 배아 품질 관련
    df['수정률']          = df['미세주입에서 생성된 배아 수'] / (df['미세주입된 난자 수'] + 1)
    df['배아_활용률']     = df['이식된 배아 수'] / (df['총 생성 배아 수'] + 1)
    df['잉여배아_존재']   = (df['저장된 배아 수'] > 0).astype(float)  # 34.1% vs 21.9%
    df['미세주입_이식비율'] = df['미세주입 배아 이식 수'] / (df['이식된 배아 수'] + 1)

    # 4. 과거 이력 관련
    시술   = df['총 시술 횟수'].map(COUNT_MAP)
    임신   = df['총 임신 횟수'].map(COUNT_MAP)
    출산   = df['총 출산 횟수'].map(COUNT_MAP)
    ivf출산 = df['IVF 출산 횟수'].map(COUNT_MAP)

    df['과거_유산횟수']  = (임신 - 출산).clip(lower=0)
    df['반복실패_여부']  = ((시술 >= 3) & (임신 == 0)).astype(int)
    df['IVF_성공이력']  = (ivf출산 > 0).astype(int)
    df['과거_시술효율']  = 임신 / (시술 + 1)

    # 5. 난자/정자 출처 이진화
    df['기증_난자'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자'] = (df['정자 출처'] == '기증 제공').astype(int)

    # 6. 배란 유도 유형 단순화 (99.9%가 기록없음/알수없음)
    df['배란유도_기록됨'] = (~df['배란 유도 유형'].isin(
        ['기록되지 않은 시행', '알 수 없음'])).astype(int)

    # 7. 배아 생성 목적 단순화
    df['현재시술_목적'] = (df['배아 생성 주요 이유'] == '현재 시술용').astype(int)

    # 8. 포배기 이식 AND 잉여배아 (교호작용) — 성공률 46.4%
    df['포배기_AND_잉여배아'] = (
        (df['배아 이식 경과일'] == 5) & (df['저장된 배아 수'] > 0)
    ).astype(float)

    # 9. 난자→배아 전환율 — 높을수록 성공률 높음
    df['난자_배아_전환율'] = df['총 생성 배아 수'] / (df['수집된 신선 난자 수'] + 1)

    # 10. 배아 저장 비율 — 0~30% 구간 성공률 38.9%
    df['배아_저장비율'] = df['저장된 배아 수'] / (df['총 생성 배아 수'] + 1)

    # 11. 미세주입 후 저장 비율 — 0~30% 구간 성공률 39.2%
    df['미세주입_저장비율'] = df['미세주입 후 저장된 배아 수'] / (df['미세주입에서 생성된 배아 수'] + 1)

    # 12. 파트너 정자 사용 비율 — 90%+ 구간 성공률 33.1%
    df['파트너정자_비율'] = df['파트너 정자와 혼합된 난자 수'] / (df['혼합된 난자 수'] + 1)

    # 13. 신선 배아 배양 시간 (배아 이식 - 난자 혼합 경과일)
    # 5일 성공률 40.5% vs 0일 12.7% → 강한 신호
    df['신선_배아_배양시간'] = (df['배아 이식 경과일'] - df['난자 혼합 경과일']).fillna(0)

    # 14. 이상적 배양기간 (3일 또는 5일)
    # 성공률 34.4% vs 비이상적 15.6%
    df['이상적_배양기간'] = df['배아 이식 경과일'].isin([3.0, 5.0]).astype(int)

    # 15. 남성/여성 불임 심각도 (개별 원인 합산)
    male_fail = ['불임 원인 - 남성 요인','불임 원인 - 정자 농도','불임 원인 - 정자 면역학적 요인',
                 '불임 원인 - 정자 운동성','불임 원인 - 정자 형태']
    female_fail = ['불임 원인 - 난관 질환','불임 원인 - 배란 장애','불임 원인 - 자궁경부 문제','불임 원인 - 자궁내막증']
    df['남성_불임_심각도'] = df[male_fail].sum(axis=1)
    df['여성_불임_심각도'] = df[female_fail].sum(axis=1)

    # 16. 실효 난자 나이 (기증 난자 사용 시 기증자 나이 반영)
    # 상관계수 -0.1632로 기존 나이 피처보다 강함
    donor_map = {'알 수 없음':-1,'만20세 이하':0,'만21-25세':1,'만26-30세':2,'만31-35세':3}
    난자기증자_코드 = df['난자 기증자 나이'].map(donor_map)
    기증_여부 = (df['난자 출처'] == '기증 제공') if df['난자 출처'].dtype == 'object' else (df['난자 출처'] == 1)
    df['실효_난자나이'] = np.where(
        기증_여부 & (난자기증자_코드 != -1),
        난자기증자_코드,
        df['시술 당시 나이'] if df['시술 당시 나이'].dtype != 'object' else df['시술 당시 나이'].map({'만18-34세':0,'만35-37세':1,'만38-39세':2,'만40-42세':3,'만43-44세':4,'만45-50세':5,'알 수 없음':2})
    )

    # 17. 고령 + 기증 난자 상쇄 (40세+ 이지만 기증 난자 → 성공률 31.6%)
    나이_코드 = df['시술 당시 나이'] if df['시술 당시 나이'].dtype != 'object' else df['시술 당시 나이'].map({'만18-34세':0,'만35-37세':1,'만38-39세':2,'만40-42세':3,'만43-44세':4,'만45-50세':5,'알 수 없음':2})
    df['고령_기증난자_상쇄'] = ((나이_코드 >= 3) & 기증_여부).astype(int)

    # 18. 혼합 난자 대비 배아 효율 (상관계수 0.1485)
    df['혼합난자_배아효율'] = df['총 생성 배아 수'] / (df['혼합된 난자 수'] + 1)

    # 19. 초고령 여부 (43세 이상, 성공률 13.6%)
    df['초고령_43이상'] = (나이_코드 >= 4).astype(int)

    # 20. 연령 x 시술 횟수 (상관계수 -0.1012)
    df['연령_x_시술횟수'] = 나이_코드 * df['총 시술 횟수']

    return df

train = add_features(train)
test  = add_features(test)

new_cols = ['포배기_이식','장기배양_이식','저반응_여부','고반응_여부','성숙_난자율',
            '수정률','배아_활용률','잉여배아_존재','미세주입_이식비율',
            '과거_유산횟수','반복실패_여부','IVF_성공이력','과거_시술효율',
            '기증_난자','기증_정자','배란유도_기록됨','현재시술_목적',
            '포배기_AND_잉여배아','난자_배아_전환율','배아_저장비율',
            '미세주입_저장비율','파트너정자_비율',
            '신선_배아_배양시간','이상적_배양기간','남성_불임_심각도','여성_불임_심각도',
            '실효_난자나이','고령_기증난자_상쇄','혼합난자_배아효율','초고령_43이상','연령_x_시술횟수']
print(f'파생변수 {len(new_cols)}개 생성 완료'    age_map_local = {'만18-34세':0,'만35-37세':1,'만38-39세':2,'만40-42세':3,'만43-44세':4,'만45-50세':5,'알 수 없음':2}
    donor_map_local = {'알 수 없음':-1,'만20세 이하':0,'만21-25세':1,'만26-30세':2,'만31-35세':3}

    나이_코드 = df['시술 당시 나이'].map(age_map_local).fillna(2)
    난자기증자_코드 = df['난자 기증자 나이'].map(donor_map_local).fillna(-1)
    기증_여부 = (df['난자 출처'] == '기증 제공')

    # 16. 실효 난자 나이
    df['실효_난자나이'] = np.where(
        기증_여부 & (난자기증자_코드 != -1),
        난자기증자_코드,
        나이_코드
    )

    # 17. 고령 + 기증 난자 상쇄
    df['고령_기증난자_상쇄'] = ((나이_코드 >= 3) & 기증_여부).astype(int)

    # 18. 혼합 난자 대비 배아 효율
    df['혼합난자_배아효율'] = df['총 생성 배아 수'] / (df['혼합된 난자 수'] + 1)

    # 19. 초고령 여부
    df['초고령_43이상'] = (나이_코드 >= 4).astype(int)

    # 20. 연령 x 시술 횟수
    시술_코드 = df['총 시술 횟수'].map(COUNT_MAP).fillna(0)
    df['연령_x_시술횟수'] = 나이_코드 * 시술_코드

    return df


## 4. 인코딩

In [ ]:
# 순서형(Ordinal) 인코딩
age_map = {'만18-34세':0,'만35-37세':1,'만38-39세':2,
           '만40-42세':3,'만43-44세':4,'만45-50세':5,'알 수 없음':-1}
donor_egg_map   = {'알 수 없음':-1,'만20세 이하':0,'만21-25세':1,'만26-30세':2,'만31-35세':3}
donor_sperm_map = {'알 수 없음':-1,'만20세 이하':0,'만21-25세':1,'만26-30세':2,
                   '만31-35세':3,'만36-40세':4,'만41-45세':5}

ordinal_mappings = {
    '시술 당시 나이': age_map,
    '총 시술 횟수': COUNT_MAP, '클리닉 내 총 시술 횟수': COUNT_MAP,
    'IVF 시술 횟수': COUNT_MAP, 'DI 시술 횟수': COUNT_MAP,
    '총 임신 횟수': COUNT_MAP, 'IVF 임신 횟수': COUNT_MAP, 'DI 임신 횟수': COUNT_MAP,
    '총 출산 횟수': COUNT_MAP, 'IVF 출산 횟수': COUNT_MAP, 'DI 출산 횟수': COUNT_MAP,
    '난자 기증자 나이': donor_egg_map,
    '정자 기증자 나이': donor_sperm_map,
}
for col, mapping in ordinal_mappings.items():
    if col in train.columns:
        train[col] = train[col].map(mapping)
        test[col]  = test[col].map(mapping)
print('순서형 인코딩 완료')

In [ ]:
# Label Encoding (명목형) — train fit → test transform
label_cols = ['시술 시기 코드','시술 유형','특정 시술 유형',
              '배란 유도 유형','배아 생성 주요 이유','난자 출처','정자 출처']
label_cols = [c for c in label_cols if c in train.columns]

label_encoders = {}
for col in label_cols:
    le = LabelEncoder()
    le.fit(train[col].astype(str))
    label_encoders[col] = le
    train[col] = le.transform(train[col].astype(str))
    test[col]  = test[col].astype(str).apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
print('Label Encoding 완료')

## 5. 최종 확인 및 저장

In [ ]:
train_ids = train['ID']
test_ids  = test['ID']

X      = train.drop(columns=['ID', TARGET])
y      = train[TARGET]
X_test = test.drop(columns=['ID'])

# 검증
assert X.isnull().sum().sum() == 0, '결측치 존재'
assert len(X.select_dtypes(exclude=np.number).columns) == 0, '비수치형 컬럼 존재'

print(f'X: {X.shape}  |  y: {y.shape}  |  X_test: {X_test.shape}')
print(f'클래스 비율: 실패 {(y==0).mean():.1%} / 성공 {(y==1).mean():.1%}')
print(f'scale_pos_weight 추천값: {(y==0).sum() / (y==1).sum():.2f}')
print('검증 완료 ✓')

In [ ]:
X.to_csv('../data/X_train.csv', index=False)
y.to_csv('../data/y_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
test_ids.to_csv('../data/test_ids.csv', index=False)
print('저장 완료 → ../data/X_train.csv, y_train.csv, X_test.csv, test_ids.csv')